In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from ev_forecasting import run_pipeline, seasonal_insights

project_dir = Path.cwd()
output_dir = project_dir / "outputs"
metrics, forecasts = run_pipeline(project_dir / "ev_charging_dataset.xlsx", output_dir, horizon=365)
daily = pd.read_csv(output_dir / "cleaned_daily_demand.csv", parse_dates=["date"], index_col="date")
print(f"Source coverage: {daily.index.min().date()} to {daily.index.max().date()}")
display(metrics.sort_values(["target", "RMSE"]))
display(daily.head())

patterns = seasonal_insights(daily, "energy_kwh")
weekday = patterns["weekday_index"].rename("Demand index").rename_axis("Weekday").reset_index()
month = patterns["month_index"].rename("Demand index").rename_axis("Month").reset_index()
display(weekday.sort_values("Demand index", ascending=False))
display(month)
px.bar(weekday, x="Weekday", y="Demand index", title="Weekday energy demand (average = 100)").show()
px.line(month, x="Month", y="Demand index", markers=True, title="Monthly energy demand (average = 100)").show()

px.line(forecasts.loc[forecasts["target"] == "energy_kwh"], x="date", y="forecast", color="model", title="Daily energy forecast (kWh)").show()
display(forecasts.head())